#### Outliers detection & removal - Using Percentile Method

In [1]:
import pandas as pd
import numpy as np

In [2]:
# ─── Inline Dataset: HDFC-style Loan Applications ───
data = {
    'applicant': ['Rahul','Priya','Suresh','Meena','Kiran',
                  'Deepak','Anita','Vikram','Pooja','Arjun',
                  'Sunita','Ravi','Neha','Amit','Divya'],
    'loan_amt_lakh': [5.5, 8.2, 12.0, 0.05, 42.0,   # 0.05 = test entry, likely error
                      18.5, 25.0, 6.8, 380.0, 15.2,  # 380 = corporate loan, outlier
                      9.4, 22.0, 11.5, 7.3, 0.03],   # 0.03 = another bad entry
    'monthly_income_k': [45, 72, 95, 30, 180,
                         110, 140, 55, 420, 88,
                         62, 125, 80, 58, 25],
    'cibil_score': [720, 750, 680, 600, 790,
                    760, 800, 710, 850, 730,
                    690, 775, 745, 700, 620],
}
df = pd.DataFrame(data)

print("Original Dataset:")
print(df[['applicant', 'loan_amt_lakh']].to_string())
print(f"\nTotal rows: {len(df)}")
print(f"Loan Amount Stats:\n{df['loan_amt_lakh'].describe().round(2)}")

Original Dataset:
   applicant  loan_amt_lakh
0      Rahul           5.50
1      Priya           8.20
2     Suresh          12.00
3      Meena           0.05
4      Kiran          42.00
5     Deepak          18.50
6      Anita          25.00
7     Vikram           6.80
8      Pooja         380.00
9      Arjun          15.20
10    Sunita           9.40
11      Ravi          22.00
12      Neha          11.50
13      Amit           7.30
14     Divya           0.03

Total rows: 15
Loan Amount Stats:
count     15.00
mean      37.57
std       95.33
min        0.03
25%        7.05
50%       11.50
75%       20.25
max      380.00
Name: loan_amt_lakh, dtype: float64


In [3]:
df

,applicant,loan_amt_lakh,monthly_income_k,cibil_score
0,Rahul,5.50,45,720
1,Priya,8.20,72,750
2,Suresh,12.00,95,680
3,Meena,0.05,30,600
4,Kiran,42.00,180,790
5,Deepak,18.50,110,760
6,Anita,25.00,140,800
7,Vikram,6.80,55,710
8,Pooja,380.00,420,850
9,Arjun,15.20,88,730


In [4]:
# ══════════════════════════════════════════
# STEP 1: Percentile Boundaries Calculate Karo
# ══════════════════════════════════════════
lower_pct = 0.05   # 5th percentile
upper_pct = 0.95   # 95th percentile

lower_bound = df['loan_amt_lakh'].quantile(lower_pct)
upper_bound = df['loan_amt_lakh'].quantile(upper_pct)

print(f"\n--- Percentile Boundaries ---")
print(f"5th  Percentile (Lower Bound): ₹{lower_bound:.2f}L")
print(f"95th Percentile (Upper Bound): ₹{upper_bound:.2f}L")
print(f"Normal range: ₹{lower_bound:.2f}L to ₹{upper_bound:.2f}L")


--- Percentile Boundaries ---
5th  Percentile (Lower Bound): ₹0.04L
95th Percentile (Upper Bound): ₹143.40L
Normal range: ₹0.04L to ₹143.40L


In [5]:
# ══════════════════════════════════════════
# STEP 2: Outliers Identify Karo
# ══════════════════════════════════════════
outlier_mask = (
    (df['loan_amt_lakh'] < lower_bound) |
    (df['loan_amt_lakh'] > upper_bound)
)

print(f"\n--- Outliers Detected ---")
print(df[outlier_mask][['applicant', 'loan_amt_lakh']])
print(f"Total outliers: {outlier_mask.sum()}")


--- Outliers Detected ---
   applicant  loan_amt_lakh
8      Pooja         380.00
14     Divya           0.03
Total outliers: 2


In [6]:
# ══════════════════════════════════════════
# STEP 3: Clean Data — Outliers Remove Karo
# ══════════════════════════════════════════
df_clean = df[~outlier_mask].copy()

print(f"\n--- Results ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_clean)}")
print(f"\nLoan Amt Mean BEFORE: ₹{df['loan_amt_lakh'].mean():.2f}L")
print(f"Loan Amt Mean AFTER:  ₹{df_clean['loan_amt_lakh'].mean():.2f}L")
print(f"\nLoan Amt Max BEFORE: ₹{df['loan_amt_lakh'].max():.2f}L")
print(f"Loan Amt Max AFTER:  ₹{df_clean['loan_amt_lakh'].max():.2f}L")


--- Results ---
Rows before: 15 | Rows after: 13

Loan Amt Mean BEFORE: ₹37.57L
Loan Amt Mean AFTER:  ₹14.11L

Loan Amt Max BEFORE: ₹380.00L
Loan Amt Max AFTER:  ₹42.00L


In [7]:
df_clean

,applicant,loan_amt_lakh,monthly_income_k,cibil_score
0,Rahul,5.50,45,720
1,Priya,8.20,72,750
2,Suresh,12.00,95,680
3,Meena,0.05,30,600
4,Kiran,42.00,180,790
5,Deepak,18.50,110,760
6,Anita,25.00,140,800
7,Vikram,6.80,55,710
9,Arjun,15.20,88,730
10,Sunita,9.40,62,690


#### Outliers detection & replace - Using Winsorization Method

In [8]:
from scipy.stats import mstats   # Method 3 ke liye

# ─── Inline Dataset: Zomato Delivery Data ───
data = {
    'rider':         ['Raju','Suresh','Amit','Pooja','Kiran',
                      'Deepak','Meena','Arjun','Sunita','Vikram',
                      'Priya','Rohit','Anita','Manoj','Kavya'],
    'delivery_min':  [28, 32, 240, 29, 31,      # 240 = accident/outlier
                      27, 185, 30, 26, 33,       # 185 = another outlier
                      29, 28, 31, 270, 27],      # 270 = extreme outlier
    'rating':        [4.5, 4.2, 3.8, 4.6, 4.4,
                      4.7, 3.5, 4.3, 4.8, 4.1,
                      4.6, 4.4, 4.3, 3.2, 4.7],
}
df = pd.DataFrame(data)

print("Original delivery_min column:")
print(df['delivery_min'].tolist())
print(f"Mean: {df['delivery_min'].mean():.1f} min")
print(f"Max:  {df['delivery_min'].max()} min")

Original delivery_min column:
[28, 32, 240, 29, 31, 27, 185, 30, 26, 33, 29, 28, 31, 270, 27]
Mean: 69.7 min
Max:  270 min


In [9]:
df

,rider,delivery_min,rating
0,Raju,28,4.5
1,Suresh,32,4.2
2,Amit,240,3.8
3,Pooja,29,4.6
4,Kiran,31,4.4
5,Deepak,27,4.7
6,Meena,185,3.5
7,Arjun,30,4.3
8,Sunita,26,4.8
9,Vikram,33,4.1


In [10]:
# ══════════════════════════════════════════
# METHOD 1: pandas clip() — Sabse Simple!
# ══════════════════════════════════════════
lower = df['delivery_min'].quantile(0.05)   # 5th percentile
upper = df['delivery_min'].quantile(0.95)   # 95th percentile

print(f"\n5th  percentile: {lower:.1f} min")
print(f"95th percentile: {upper:.1f} min")

# clip() = Winsorization! Values ko boundaries pe clamp karta hai
df['delivery_winsorized'] = df['delivery_min'].clip(lower=lower, upper=upper)

print("\nAfter Winsorization (clip):")
print(df['delivery_winsorized'].tolist())
print(f"Mean after: {df['delivery_winsorized'].mean():.1f} min")
print(f"Max after:  {df['delivery_winsorized'].max():.1f} min")
# 240, 185, 270 sab → upper boundary pe cap ho gaye!


5th  percentile: 26.7 min
95th percentile: 249.0 min

After Winsorization (clip):
[28.0, 32.0, 240.0, 29.0, 31.0, 27.0, 185.0, 30.0, 26.7, 33.0, 29.0, 28.0, 31.0, 248.99999999999997, 27.0]
Mean after: 68.4 min
Max after:  249.0 min


In [11]:
df

,rider,delivery_min,rating,delivery_winsorized
0,Raju,28,4.5,28.0
1,Suresh,32,4.2,32.0
2,Amit,240,3.8,240.0
3,Pooja,29,4.6,29.0
4,Kiran,31,4.4,31.0
5,Deepak,27,4.7,27.0
6,Meena,185,3.5,185.0
7,Arjun,30,4.3,30.0
8,Sunita,26,4.8,26.7
9,Vikram,33,4.1,33.0


In [12]:
# ══════════════════════════════════════════
# METHOD 2: numpy clip() — Same result
# ══════════════════════════════════════════
df['delivery_np_clip'] = np.clip(df['delivery_min'], lower, upper)
# df['delivery_winsorized'] aur df['delivery_np_clip'] same honge!

df

,rider,delivery_min,rating,delivery_winsorized,delivery_np_clip
0,Raju,28,4.5,28.0,28.0
1,Suresh,32,4.2,32.0,32.0
2,Amit,240,3.8,240.0,240.0
3,Pooja,29,4.6,29.0,29.0
4,Kiran,31,4.4,31.0,31.0
5,Deepak,27,4.7,27.0,27.0
6,Meena,185,3.5,185.0,185.0
7,Arjun,30,4.3,30.0,30.0
8,Sunita,26,4.8,26.7,26.7
9,Vikram,33,4.1,33.0,33.0


In [13]:
# ══════════════════════════════════════════
# METHOD 3: scipy winsorize() — Direct
# ══════════════════════════════════════════
from scipy.stats.mstats import winsorize

df['delivery_scipy'] = winsorize(
    df['delivery_min'],
    limits=[0.05, 0.05]   # bottom 5% aur top 5% winsorize karo
)
print("\nDataset size BEFORE:", len(df))
print("Dataset size AFTER:", len(df))   # Same! Rows nahi gayi!



Dataset size BEFORE: 15
Dataset size AFTER: 15


In [14]:
df

,rider,delivery_min,rating,delivery_winsorized,delivery_np_clip,delivery_scipy
0,Raju,28,4.5,28.0,28.0,28
1,Suresh,32,4.2,32.0,32.0,32
2,Amit,240,3.8,240.0,240.0,240
3,Pooja,29,4.6,29.0,29.0,29
4,Kiran,31,4.4,31.0,31.0,31
5,Deepak,27,4.7,27.0,27.0,27
6,Meena,185,3.5,185.0,185.0,185
7,Arjun,30,4.3,30.0,30.0,30
8,Sunita,26,4.8,26.7,26.7,26
9,Vikram,33,4.1,33.0,33.0,33
